In [1]:
from pathlib import Path
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, log_loss, accuracy_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "features"
TRAIN_DIR = PROJECT_ROOT / "data" / "processed" / "train"
TEST_DIR = PROJECT_ROOT / "data" / "processed" / "student_test"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "model_results"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(FEATURE_DIR / "train_candidates.csv")
student_test = pd.read_csv(FEATURE_DIR / "student_test_candidates.csv")
FEATURES = pd.read_csv(FEATURE_DIR / "feature_columns.csv")["feature"].tolist()
train["trip_id"] = train["trip_id"].astype(str).str.zfill(8)
student_test["trip_id"] = student_test["trip_id"].astype(str).str.zfill(8)
train["target"] = train["target"].astype(int)
GROUP = ["trip_id", "observation_pos"]

# Remove observations whose correct arc is absent from the candidate list.
positive = train.groupby(GROUP)["target"].transform("max") == 1
train = train.loc[positive].copy()
train[FEATURES] = train[FEATURES].replace([np.inf, -np.inf], np.nan)
student_test[FEATURES] = student_test[FEATURES].replace([np.inf, -np.inf], np.nan)
print("Training rows:", len(train), "| trips:", train.trip_id.nunique())
print("Student-test rows:", len(student_test), "| trips:", student_test.trip_id.nunique())
print("Features:", FEATURES)

Training rows: 2590102 | trips: 80
Student-test rows: 275425 | trips: 10
Features: ['candidate_rank', 'distance_m', 'distance_to_projected_point_m', 'segment_length_m', 'gps_heading_deg', 'road_heading_deg', 'heading_difference_deg', 'projection_fraction', 'used_fallback', 'search_radius_m']


In [2]:
try:
    from xgboost import XGBClassifier
except ImportError:
    %pip install xgboost
    from xgboost import XGBClassifier

TRAIN_MAX_ROWS = 250_000
SVM_MAX_ROWS = 15_000

def make_training_sample(frame, max_rows=TRAIN_MAX_ROWS, seed=42):
    pos = frame[frame.target == 1]
    neg = frame[frame.target == 0]
    if len(pos) == 0 or len(neg) == 0:
        return frame
    if len(frame) <= max_rows:
        return frame
    # Keep both classes in every sampled fit; cap each class to half the sample.
    per_class = max_rows // 2
    pos_n = min(len(pos), per_class)
    neg_n = min(len(neg), max_rows - pos_n)
    if pos_n + neg_n < max_rows:
        extra = max_rows - pos_n - neg_n
        if len(pos) > pos_n:
            add = min(extra, len(pos) - pos_n)
            pos_n += add
            extra -= add
        if extra and len(neg) > neg_n:
            neg_n += min(extra, len(neg) - neg_n)
    sampled = pd.concat([
        pos.sample(pos_n, random_state=seed),
        neg.sample(neg_n, random_state=seed)
    ])
    return sampled.sample(frac=1, random_state=seed).reset_index(drop=True)

def make_models(use_cuda=True):
    xgb_device = 'cuda' if use_cuda else 'cpu'
    return {
        'Logistic Regression': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=500, class_weight='balanced')),
        'Random Forest': make_pipeline(SimpleImputer(strategy='median'), RandomForestClassifier(n_estimators=80, max_depth=20, min_samples_leaf=3, max_samples=0.8, class_weight='balanced_subsample', n_jobs=2, random_state=42)),
        'Extra Trees': make_pipeline(SimpleImputer(strategy='median'), ExtraTreesClassifier(n_estimators=80, max_depth=20, min_samples_leaf=3, class_weight='balanced', n_jobs=2, random_state=42)),
        'XGBoost': make_pipeline(SimpleImputer(strategy='median'), XGBClassifier(n_estimators=120, max_depth=6, learning_rate=0.08, subsample=0.85, colsample_bytree=0.85, objective='binary:logistic', eval_metric='logloss', tree_method='hist', device=xgb_device, n_jobs=4, random_state=42)),
        'RBF SVM': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), SVC(C=2.0, kernel='rbf', probability=True, class_weight='balanced', cache_size=1000, random_state=42))
    }

def fit_model(name, model, frame, features=FEATURES):
    cap = SVM_MAX_ROWS if name == 'RBF SVM' else TRAIN_MAX_ROWS
    fit_frame = make_training_sample(frame, cap)
    classes = np.unique(fit_frame.target)
    if len(classes) < 2:
        raise ValueError(f"{name}: this training fold contains only class {classes.tolist()}. Use stratified group folds; a classifier cannot learn from one class.")
    model.fit(fit_frame[features], fit_frame.target)
    return model

models = make_models(use_cuda=True)
print('Training cap per fit:', TRAIN_MAX_ROWS, '| SVM cap:', SVM_MAX_ROWS)
print('Models:', list(models))

Training cap per fit: 250000 | SVM cap: 15000
Models: ['Logistic Regression', 'Random Forest', 'Extra Trees', 'XGBoost', 'RBF SVM']


In [3]:
def svm_sample(frame, max_rows=SVM_MAX_ROWS):
    return make_training_sample(frame, max_rows)

# Stratify by candidate target while keeping entire trips together.
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
cv_rows = []
for name in models:
    fold_true, fold_pred = [], []
    for fold, (tr_idx, va_idx) in enumerate(cv.split(train, train.target, groups=train.trip_id), 1):
        fold_train = train.iloc[tr_idx]
        fold_val = train.iloc[va_idx]
        if fold_train.target.nunique() < 2:
            raise ValueError(f"Fold {fold} has only one training class. Check positive labels per trip in train_candidates.csv.")
        model = make_models(use_cuda=True)[name]
        try:
            fit_model(name, model, fold_train)
        except Exception as e:
            if name == 'XGBoost':
                print('CUDA unavailable; retrying XGBoost on CPU:', str(e)[:220])
                model = make_models(use_cuda=False)[name]
                fit_model(name, model, fold_train)
            else:
                raise
        pred = model.predict(fold_val[FEATURES])
        fold_true.extend(fold_val.target.tolist())
        fold_pred.extend(pred.tolist())
        print(name, 'fold', fold, 'complete')
        del model, fold_train, fold_val, pred
        import gc; gc.collect()
    cv_rows.append({'model': name, 'cv_macro_f1': f1_score(fold_true, fold_pred, average='macro', zero_division=0)})
    print(name, 'CV Macro F1:', round(cv_rows[-1]['cv_macro_f1'], 4))
    del fold_true, fold_pred
    import gc; gc.collect()

cv_df = pd.DataFrame(cv_rows)
display(cv_df.sort_values('cv_macro_f1', ascending=False))

# A group-aware 20% holdout, also stratified by target.
holdout_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=123)
tr_idx, va_idx = next(holdout_splitter.split(train, train.target, groups=train.trip_id))
fit_data, val_data = train.iloc[tr_idx].copy(), train.iloc[va_idx].copy()
print('Fit trips:', fit_data.trip_id.nunique(), 'Validation trips:', val_data.trip_id.nunique())
print('Fit classes:', fit_data.target.value_counts().to_dict())


Logistic Regression fold 1 complete
Logistic Regression fold 2 complete
Logistic Regression fold 3 complete
Logistic Regression CV Macro F1: 0.7283
Random Forest fold 1 complete
Random Forest fold 2 complete
Random Forest fold 3 complete
Random Forest CV Macro F1: 0.804
Extra Trees fold 1 complete
Extra Trees fold 2 complete
Extra Trees fold 3 complete
Extra Trees CV Macro F1: 0.7978


c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\core.py:569: UserWarning: [15:21:22] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


XGBoost fold 1 complete
XGBoost fold 2 complete
XGBoost fold 3 complete
XGBoost CV Macro F1: 0.8057
RBF SVM fold 1 complete
RBF SVM fold 2 complete
RBF SVM fold 3 complete
RBF SVM CV Macro F1: 0.7795


,model,cv_macro_f1
3,XGBoost,0.805749
1,Random Forest,0.803997
2,Extra Trees,0.797758
4,RBF SVM,0.779495
0,Logistic Regression,0.728330


Fit trips: 64 Validation trips: 16
Fit classes: {0: 1905976, 1: 165506}


In [4]:
def route_continuity(predictions, graph_dir):
    values = []
    for trip_id, group in predictions.groupby("trip_id"):
        arc_path = graph_dir / trip_id / "arcs.csv"
        if not arc_path.exists():
            continue
        arcs = pd.read_csv(arc_path)
        if not {"from_node_id", "to_node_id"}.issubset(arcs.columns):
            continue
        arcs = arcs.reset_index(drop=True)
        best = group.sort_values(["observation_pos", "score"], ascending=[True, False]).drop_duplicates("observation_pos")
        ids = pd.to_numeric(best.arc_id, errors="coerce").dropna().astype(int).tolist()
        pairs = [(a, b) for a, b in zip(ids[:-1], ids[1:]) if a < len(arcs) and b < len(arcs)]
        if pairs:
            connected = [int(arcs.iloc[a].to_node_id) == int(arcs.iloc[b].from_node_id) for a, b in pairs]
            values.extend(connected)
    return float(np.mean(values)) if values else np.nan

def best_candidate_frame(frame, scores):
    out = frame[GROUP + ["arc_id", "target"]].copy()
    out["score"] = scores
    return out.sort_values(["trip_id", "observation_pos", "score"], ascending=[True, True, False]).drop_duplicates(GROUP)

rows, fitted, val_score_frames = [], {}, []
for name in models:
    model = make_models(use_cuda=True)[name]
    start = time.perf_counter()
    try:
        fit_model(name, model, fit_data)
    except Exception as e:
        if name == "XGBoost":
            print("CUDA unavailable during final fit; using CPU:", str(e)[:220])
            model = make_models(use_cuda=False)[name]
            fit_model(name, model, fit_data)
        else:
            raise
    fit_seconds = time.perf_counter() - start
    start = time.perf_counter()
    probabilities = model.predict_proba(val_data[FEATURES])[:, 1]
    inference_seconds = time.perf_counter() - start
    predicted = (probabilities >= 0.5).astype(int)
    best = best_candidate_frame(val_data, probabilities)
    top1 = float(best.target.mean())
    continuity = route_continuity(best, TRAIN_DIR)
    rows.append({"model": name, "cv_macro_f1": cv_df.loc[cv_df.model == name, "cv_macro_f1"].iloc[0], "top1_accuracy": top1, "route_continuity": continuity, "log_loss": log_loss(val_data.target, np.clip(probabilities, 1e-7, 1-1e-7)), "inference_time_sec": inference_seconds, "fit_time_sec": fit_seconds})
    fitted[name] = model
    pred_frame = val_data[GROUP + ["arc_id", "target"]].copy()
    pred_frame["score"] = probabilities
    pred_frame["model"] = name
    val_score_frames.append(pred_frame)
    print(name, "Top-1:", round(top1, 4), "fit(s):", round(fit_seconds, 1), "inference(s):", round(inference_seconds, 2))
    import gc; gc.collect()

comparison = pd.DataFrame(rows).sort_values(["cv_macro_f1", "top1_accuracy"], ascending=False)
display(comparison)

Logistic Regression Top-1: 0.7364 fit(s): 0.9 inference(s): 0.22
Random Forest Top-1: 0.703 fit(s): 23.6 inference(s): 1.72
Extra Trees Top-1: 0.728 fit(s): 5.9 inference(s): 2.09
XGBoost Top-1: 0.6917 fit(s): 1.4 inference(s): 0.25
RBF SVM Top-1: 0.7008 fit(s): 6.0 inference(s): 51.43


,model,cv_macro_f1,top1_accuracy,route_continuity,log_loss,inference_time_sec,fit_time_sec
3,XGBoost,0.805749,0.691731,0.317868,0.197450,0.254439,1.413867
1,Random Forest,0.803997,0.703005,0.317498,0.196793,1.719146,23.616460
2,Extra Trees,0.797758,0.727995,0.327098,0.190317,2.091628,5.859205
4,RBF SVM,0.779495,0.700809,0.317498,0.231299,51.430647,5.981948
0,Logistic Regression,0.728330,0.736358,0.322187,0.250608,0.216581,0.871276


In [5]:
best_name = comparison.iloc[0].model
best_model = fitted[best_name]
test_prob = best_model.predict_proba(student_test[FEATURES])[:, 1]
test_pred = student_test[GROUP + ["arc_id"]].copy()
test_pred["score"] = test_prob
test_pred = test_pred.sort_values(["trip_id", "observation_pos", "score"], ascending=[True, True, False]).drop_duplicates(GROUP)
test_pred = test_pred.rename(columns={"arc_id": "predicted_arc_id"})

label_path = FEATURE_DIR / "student_test_provisional_labels.csv"
student_f1 = np.nan
if label_path.exists():
    labels = pd.read_csv(label_path)
    labels["trip_id"] = labels["trip_id"].astype(str).str.zfill(8)
    label_col = "label_arc_id" if "label_arc_id" in labels.columns else None
    if label_col and "observation_pos" in labels.columns:
        truth = labels[GROUP + [label_col]].drop_duplicates(GROUP)
        scored = test_pred.merge(truth, on=GROUP, how="inner")
        if len(scored):
            student_f1 = f1_score(scored[label_col].astype(int), scored.predicted_arc_id.astype(int), average="macro", zero_division=0)

comparison["student_test_macro_f1"] = student_f1
comparison["algorithm_family"] = comparison.model.map({"Logistic Regression":"Linear", "Random Forest":"Bagging ensemble", "Extra Trees":"Bagging ensemble", "XGBoost":"Gradient boosting", "RBF SVM":"Kernel method"})
comparison["key_configuration"] = comparison.model.map({"Logistic Regression":"L2, balanced", "Random Forest":"80 trees, depth 20, balanced", "Extra Trees":"80 trees, depth 20, balanced", "XGBoost":"120 trees, depth 6, CUDA if available", "RBF SVM":"RBF, C=2, sampled fit"})
comparison = comparison[["model", "algorithm_family", "key_configuration", "cv_macro_f1", "top1_accuracy", "route_continuity", "student_test_macro_f1", "log_loss", "inference_time_sec"]]
comparison.to_csv(OUTPUT_DIR / "model_comparison_consolidated.csv", index=False)

print("Best model:", best_name)
print("Student-test F1 is provisional:", student_f1)
display(comparison)

Best model: XGBoost
Student-test F1 is provisional: 0.6341861584716678


,model,algorithm_family,key_configuration,cv_macro_f1,top1_accuracy,route_continuity,student_test_macro_f1,log_loss,inference_time_sec
3,XGBoost,Gradient boosting,"120 trees, depth 6, CUDA if available",0.805749,0.691731,0.317868,0.634186,0.197450,0.254439
1,Random Forest,Bagging ensemble,"80 trees, depth 20, balanced",0.803997,0.703005,0.317498,0.634186,0.196793,1.719146
2,Extra Trees,Bagging ensemble,"80 trees, depth 20, balanced",0.797758,0.727995,0.327098,0.634186,0.190317,2.091628
4,RBF SVM,Kernel method,"RBF, C=2, sampled fit",0.779495,0.700809,0.317498,0.634186,0.231299,51.430647
0,Logistic Regression,Linear,"L2, balanced",0.728330,0.736358,0.322187,0.634186,0.250608,0.216581


In [6]:
joblib.dump(best_model, MODEL_DIR / "best_candidate_model.joblib")
(OUTPUT_DIR / "best_model_name.txt").write_text(best_name, encoding="utf-8")
pd.concat(val_score_frames, ignore_index=True).to_csv(OUTPUT_DIR / "validation_candidate_scores.csv", index=False)
test_pred.to_csv(OUTPUT_DIR / "student_test_candidate_scores.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "model_comparison_consolidated.csv", index=False)

print("Saved model:", MODEL_DIR / "best_candidate_model.joblib")
print("Saved comparison:", OUTPUT_DIR / "model_comparison_consolidated.csv")
print("Saved student-test scores:", OUTPUT_DIR / "student_test_candidate_scores.csv")

Saved model: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\models\best_candidate_model.joblib
Saved comparison: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\model_results\model_comparison_consolidated.csv
Saved student-test scores: c:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\model_results\student_test_candidate_scores.csv


### Notes
- This version caps each model fit at 250,000 rows (15,000 for RBF SVM) and uses a reproducible class-aware sample to reduce RAM use. All validation predictions are still evaluated on the full held-out fold.
- Random Forest and Extra Trees use two CPU workers to avoid parallel memory spikes.
- CUDA acceleration is attempted for XGBoost only; other scikit-learn models run on CPU.
- CV Macro F1 is candidate-level binary classification F1. Top-1 accuracy measures whether the correct arc ranks first within each observation.
- Route continuity is graph-edge connectivity between consecutive selected arcs; it is not a substitute for full sequence map matching.
- Student-test Macro F1 is against provisional labels, not ground truth.

- Uses stratified group folds: target classes are balanced across folds without splitting observations from the same trip across train/validation. This fixes the one-class SVM fold error.
